# Differential Geometries and Measurable Modes

Notebook 23 showed that differential readout removes some disturbances and preserves others.

Notebook 29 asks:

**How does sensor arrangement determine which disturbances are removable?**

This refined version treats each geometry as a spatial filter and measures wavelength response using RMS response over phase. That removes phase-sampling artifacts and makes the detector-design lesson clearer.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("..")
FIGURES = ROOT / "figures"
DATA = ROOT / "data"

FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

rng = np.random.default_rng(42)


## Simulated disturbance field

We begin with a one-dimensional disturbance field containing long-, medium-, and short-wavelength components.

The sensors sample this field at four positions. Different measurement geometries combine those samples differently.


In [ ]:
x = np.linspace(0, 100, 1000)

field = (
    1.0 * np.sin(2 * np.pi * x / 120)
    + 0.5 * np.sin(2 * np.pi * x / 40)
    + 0.2 * np.sin(2 * np.pi * x / 10)
)

sensor_positions = {
    "A": 10,
    "B": 40,
    "C": 70,
    "D": 100
}

geometry_weights = {
    "single (A)": {"A": 1},
    "pair (A-B)": {"A": 1, "B": -1},
    "triple (A-2B+C)": {"A": 1, "B": -2, "C": 1},
    "four-point (A-B-C+D)": {"A": 1, "B": -1, "C": -1, "D": 1},
}


def field_at(position):
    return np.interp(position, x, field)


def weighted_response(values, weights):
    return sum(weights[name] * values[name] for name in weights)


In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(x, field, label="disturbance field")

for name, pos in sensor_positions.items():
    y = field_at(pos)
    plt.axvline(pos, linestyle="--", alpha=0.5)
    plt.scatter([pos], [y], zorder=3)
    plt.text(pos, y + 0.15, name, ha="center")

plt.xlabel("position")
plt.ylabel("field amplitude")

plt.title("Spatial disturbance field sampled by sensors")

plt.tight_layout()

plt.savefig(
    FIGURES / "29_spatial_field_refined.png",
    dpi=200
)

plt.show()


## Response to the example field

The same field produces different responses depending on the measurement geometry.

This is the first detector-design idea: geometry is not passive. It determines which spatial modes become visible.


In [ ]:
sample_values = {
    name: field_at(pos)
    for name, pos in sensor_positions.items()
}

rows = []

for geometry, weights in geometry_weights.items():
    response = weighted_response(sample_values, weights)
    rows.append({
        "geometry": geometry,
        "response": response,
        "absolute_response": abs(response)
    })

response_df = pd.DataFrame(rows)
response_df


In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    response_df["geometry"],
    response_df["absolute_response"]
)

plt.ylabel("Absolute response")
plt.title("Different geometries produce different responses to the same field")
plt.xticks(rotation=15, ha="right")

plt.tight_layout()

plt.savefig(
    FIGURES / "29_geometry_response_refined.png",
    dpi=200
)

plt.show()


## RMS wavelength response

The first version of this notebook measured response at one sinusoidal phase. That created oscillatory artifacts.

Here we sweep the disturbance wavelength and average the response over phase.

For each wavelength, the field is:

\[
f(x) = \sin(2\pi x / \lambda + \theta)
\]

and the plotted value is the RMS response over many phases \(\theta\).


In [ ]:
wavelengths = np.logspace(0, 3, 180)
phases = np.linspace(0, 2 * np.pi, 160, endpoint=False)


def sinusoid_values(wavelength, phase):
    return {
        name: np.sin(2 * np.pi * pos / wavelength + phase)
        for name, pos in sensor_positions.items()
    }


def rms_response_for_geometry(wavelength, weights):
    responses = []

    for phase in phases:
        values = sinusoid_values(wavelength, phase)
        responses.append(weighted_response(values, weights))

    responses = np.array(responses)
    return np.sqrt(np.mean(responses ** 2))


rows = []

for wavelength in wavelengths:
    for geometry, weights in geometry_weights.items():
        rows.append({
            "wavelength": wavelength,
            "geometry": geometry,
            "rms_response": rms_response_for_geometry(wavelength, weights)
        })

sweep = pd.DataFrame(rows)


In [ ]:
pair = sweep[sweep["geometry"] == "pair (A-B)"]

plt.figure(figsize=(8, 4))

plt.loglog(
    pair["wavelength"],
    pair["rms_response"]
)

baseline_ab = abs(sensor_positions["B"] - sensor_positions["A"])

plt.axvline(
    baseline_ab,
    linestyle="--",
    alpha=0.6,
    label="A-B baseline"
)

plt.xlabel("Disturbance wavelength")
plt.ylabel("RMS pair response")

plt.title("Pair geometry suppresses wavelengths longer than its baseline")
plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "29_pair_wavelength_response_refined.png",
    dpi=200
)

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

for geometry, group in sweep.groupby("geometry"):
    plt.loglog(
        group["wavelength"],
        group["rms_response"],
        label=geometry
    )

plt.xlabel("Disturbance wavelength")
plt.ylabel("RMS response")

plt.title("Differential geometries reject long-wavelength common structure")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "29_geometry_comparison_refined.png",
    dpi=200
)

plt.show()


## Normalized long-wavelength rejection

To compare shapes rather than raw scale, normalize each geometry by its short-wavelength response.

This highlights the main design rule:

**Higher-order differential geometries reject long-wavelength structure more strongly.**


In [ ]:
normalized_rows = []

for geometry, group in sweep.groupby("geometry"):
    group = group.sort_values("wavelength").copy()
    reference = group["rms_response"].iloc[0]

    group["normalized_response"] = group["rms_response"] / reference
    normalized_rows.append(group)

normalized = pd.concat(normalized_rows, ignore_index=True)


In [ ]:
plt.figure(figsize=(9, 5))

for geometry, group in normalized.groupby("geometry"):
    plt.loglog(
        group["wavelength"],
        group["normalized_response"],
        label=geometry
    )

plt.xlabel("Disturbance wavelength")
plt.ylabel("Normalized RMS response")

plt.title("Geometry determines long-wavelength rejection")

plt.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "29_normalized_geometry_rejection.png",
    dpi=200
)

plt.show()


## Geometry summary

The weighted sums act like discrete spatial derivatives.

A single sensor measures a local field value.

A pair measures a first difference.

A triple baseline measures a second difference.

A four-point contrast measures a higher-order spatial contrast.


In [ ]:
summary = pd.DataFrame({
    "geometry": [
        "single (A)",
        "pair (A-B)",
        "triple (A-2B+C)",
        "four-point (A-B-C+D)"
    ],
    "measurement": [
        "field value",
        "first difference",
        "second difference",
        "four-point contrast"
    ],
    "acts_like": [
        "zeroth order",
        "first spatial difference",
        "second spatial difference",
        "higher-order spatial contrast"
    ],
    "best_for": [
        "local value",
        "gradients / slopes",
        "curvature-like structure",
        "spatial contrast"
    ],
    "strongly_rejects": [
        "nothing",
        "common offsets and very long wavelengths",
        "common offsets, trends, and longer wavelengths",
        "selected common and gradient-like modes"
    ]
})

summary.to_csv(
    DATA / "29_geometry_summary_refined.csv",
    index=False
)

summary


In [ ]:
sweep.to_csv(
    DATA / "29_geometry_wavelength_sweep_refined.csv",
    index=False
)

normalized.to_csv(
    DATA / "29_geometry_wavelength_sweep_normalized.csv",
    index=False
)


# Lesson

Differential sensing is not only about cancelling noise.

Sensor geometry determines which spatial modes are visible.

A measurement geometry acts like a filter.

Higher-order differential geometries reject long-wavelength common structure more strongly and emphasize spatial variation.

Geometry creates context.

Context determines which signals survive measurement.

Notebook 37 will connect this toy geometry directly to an atom-interferometer-style differential readout.
